# Final MARL


In [1]:

%pip install -q pettingzoo==1.24.3 gymnasium==0.29.1 numpy==1.26.4 torch==2.3.1 matplotlib==3.8.4


Note: you may need to restart the kernel to use updated packages.


# warehouse robot

In [ ]:
import os
import random
import numpy as np
import torch

# simple config
if not os.path.exists("results"):
    os.mkdir("results")

def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


Using device: cpu


In [ ]:
import numpy as np
from pettingzoo.utils import ParallelEnv
from gymnasium.spaces import Discrete, Box


class WarehouseParallelEnv(ParallelEnv):

    metadata = {
        "name": "Warehouse",
        "render_modes": ["human"]
    }

    def __init__(self, grid_size=6, num_agents=3, max_steps=200, seed=42):
        super().__init__()

        self.grid_size = grid_size
        self._n_agents = num_agents
        self.max_steps = max_steps

        self.rng = np.random.RandomState(seed)

        self.possible_agents = [f"agent_{i}" for i in range(num_agents)]
        self.agents = self.possible_agents.copy()

        # action: 0 up, 1 down, 2 left, 3 right, 4 pick, 5 drop
        self.action_spaces = {
            agent: Discrete(6) for agent in self.possible_agents
        }

        self.observation_spaces = {
            agent: Box(
                low=0.0,
                high=1.0,
                shape=(7,),
                dtype=np.float32
            )
            for agent in self.possible_agents
        }

        self._agent_pos = {}
        self._pickup_pos = {}
        self._drop_pos = {}
        self._carrying = {}
        self._picked_once = {}
        self._delivered = {}
        self._steps = 0


    def _sample_empty(self, occupied):
        while True:
            r = int(self.rng.randint(0, self.grid_size))
            c = int(self.rng.randint(0, self.grid_size))
            if (r, c) not in occupied:
                occupied.add((r, c))
                return (r, c)

    def _obs_agent(self, agent):
        r, c = self._agent_pos[agent]
        pr, pc = self._pickup_pos[agent]
        dr, dc = self._drop_pos[agent]
        carry_flag = float(self._carrying[agent])

        denom = max(1, self.grid_size - 1)
        rr = r / denom
        cc = c / denom
        prr = pr / denom
        pcc = pc / denom
        drr = dr / denom
        dcc = dc / denom

        vec = np.array(
            [rr, cc, prr, pcc, drr, dcc, carry_flag],
            dtype=np.float32
        )
        return vec

    def _get_obs_all(self):
        obs = {}
        for a in self.agents:
            obs[a] = self._obs_agent(a)
        return obs


    def reset(self, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.RandomState(seed)

        self.agents = self.possible_agents.copy()
        self._steps = 0

        occupied = set()

        for a in self.possible_agents:
            self._agent_pos[a] = self._sample_empty(occupied)

        for a in self.possible_agents:
            self._pickup_pos[a] = self._sample_empty(occupied)
            self._drop_pos[a] = self._sample_empty(occupied)
            self._carrying[a] = 0
            self._picked_once[a] = False
            self._delivered[a] = False

        observations = self._get_obs_all()
        infos = {a: {} for a in self.agents}
        return observations, infos

    def step(self, actions):
        if not self.agents:
            return {}, {}, {}, {}, {}

        self._steps += 1

        next_pos = {}
        for a in self.agents:
            act = actions.get(a, 0)
            r, c = self._agent_pos[a]

            if act == 0:      # up
                r = max(0, r - 1)
            elif act == 1:    # down
                r = min(self.grid_size - 1, r + 1)
            elif act == 2:    # left
                c = max(0, c - 1)
            elif act == 3:    # right
                c = min(self.grid_size - 1, c + 1)

            next_pos[a] = (r, c)

        rewards = {a: -1.0 for a in self.agents}

        cell_to_agents = {}
        for a, p in next_pos.items():
            if p not in cell_to_agents:
                cell_to_agents[p] = []
            cell_to_agents[p].append(a)

        for cell, group in cell_to_agents.items():
            if len(group) > 1:
                for a in group:
                    next_pos[a] = self._agent_pos[a]
                    rewards[a] += -20.0

        for a in self.agents:
            self._agent_pos[a] = next_pos[a]

        for a in self.agents:
            act = actions.get(a, 0)

            if (
                act == 4
                and self._carrying[a] == 0
                and self._agent_pos[a] == self._pickup_pos[a]
                and not self._delivered[a]
            ):
                self._carrying[a] = 1
                if not self._picked_once[a]:
                    rewards[a] += 25.0
                    self._picked_once[a] = True

            elif (
                act == 5
                and self._carrying[a] == 1
                and self._agent_pos[a] == self._drop_pos[a]
            ):
                self._carrying[a] = 0
                self._delivered[a] = True
                rewards[a] += 100.0

        all_delivered = all(self._delivered[a] for a in self.agents)
        terminations = {}
        truncations = {}

        for a in self.agents:
            terminations[a] = bool(all_delivered)
            truncations[a] = (self._steps >= self.max_steps)

        if all(terminations.values()) or all(truncations.values()):
            self.agents = []

        observations = self._get_obs_all()
        infos = {a: {} for a in observations.keys()}

        return observations, rewards, terminations, truncations, infos


In [ ]:
class MultiAgentIDQN:
    def __init__(self, env, lr=2e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay_steps=50_000,
                 target_update_interval=1000, buffer_size=200_000, batch_size=128):
        self.env = env
        self.a_list = env.possible_agents
        self.obs_dim = {a: env.observation_spaces[a].shape[0] for a in self.a_list}
        self.act_num = {a: env.action_spaces[a].n for a in self.a_list}
        self.ag = {a: DQNAgent(self.obs_dim[a], self.act_num[a], lr=lr, gamma=gamma,
                                eps_start=eps_start, eps_end=eps_end, eps_decay_steps=eps_decay_steps,
                                target_update_interval=target_update_interval,
                                buffer_size=buffer_size, batch_size=batch_size) for a in self.a_list}
        self.step = 0
    def select_actions(self, obs_dict):
        if not obs_dict:
            return {}
        eps = np.mean([self.ag[a].epsilon(self.step) for a in obs_dict.keys()])
        return {a: self.ag[a].act(obs_dict[a], eps) for a in obs_dict.keys()}
    def store_transitions(self, obs, actions, rewards, next_obs, dones):
        for a in obs.keys():
            self.ag[a].push(obs[a], actions.get(a, 0), rewards.get(a, 0.0), next_obs.get(a, obs[a]), bool(dones.get(a, False)))
    def train_step(self):
        return float(np.mean([self.ag[a].train_step() for a in self.a_list]))

def train_multiagent(env_builder, num_episodes=500, max_steps=200,
                      lr=2e-4, gamma=0.99,
                      eps_start=1.0, eps_end=0.05, eps_decay_steps=50_000,
                      target_update_interval=1000, buffer_size=200_000, batch_size=128,
                      seed=42, log_every=10, label="run"):
    set_seed(seed)
    env = env_builder()
    algo = MultiAgentIDQN(env, lr, gamma, eps_start, eps_end, eps_decay_steps,
                          target_update_interval, buffer_size, batch_size)
    ep_returns, all_losses, succ_list = [], [], []
    for ep in range(1, num_episodes + 1):
        obs, _ = env.reset(seed=seed + ep)
        ep_r, succ = 0.0, 0
        for t in range(max_steps):
            acts = algo.select_actions(obs)
            if not acts:
                break
            nobs, rews, terms, truncs, infos = env.step(acts)
            dones = {a: terms[a] or truncs[a] for a in env.possible_agents}
            algo.store_transitions(obs, acts, rews, nobs, dones)
            all_losses.append(algo.train_step())
            ep_r += float(np.mean(list(rews.values())))
            obs = nobs
            algo.step += 1
            if all(dones.values()):
                if any(terms.values()):
                    succ = 1
                break
        ep_returns.append(ep_r)
        succ_list.append(succ)
        if ep % log_every == 0:
            print(f"[{label}] ep {ep}/{num_episodes}  avgR={np.mean(ep_returns[-log_every:]):.2f}  succ={np.mean(succ_list[-log_every:]):.2f}")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep_returns); ax[0].set_title(f"Reward ({label})")
    ax[1].plot(np.convolve(succ_list, np.ones(20)/20, mode='same') if len(succ_list)>0 else [])
    fig.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR, f"training_curves_{label}.png"), dpi=150); plt.close(fig)
    return {"episode_rewards": ep_returns, "losses": all_losses, "success_rates": succ_list}


In [ ]:
class ReplayBuffer:
    def __init__(self, cap, obs_dim):
        self.cap = int(cap)

        self.obs = np.zeros((self.cap, obs_dim), dtype=np.float32)
        self.act = np.zeros((self.cap,), dtype=np.int64)
        self.rew = np.zeros((self.cap,), dtype=np.float32)
        self.nobs = np.zeros((self.cap, obs_dim), dtype=np.float32)
        self.done = np.zeros((self.cap,), dtype=np.float32)

        self.ptr = 0
        self.sz = 0

    def push(self, o, a, r, no, d):
        i = self.ptr

        self.obs[i] = o
        self.act[i] = a
        self.rew[i] = r
        self.nobs[i] = no
        self.done[i] = 1.0 if d else 0.0

        self.ptr = (self.ptr + 1) % self.cap
        self.sz = min(self.sz + 1, self.cap)

    def sample(self, bs):
        idx = np.random.randint(0, self.sz, size=bs)
        obs = torch.as_tensor(self.obs[idx], dtype=torch.float32, device=DEVICE)
        act = torch.as_tensor(self.act[idx], dtype=torch.long, device=DEVICE)
        rew = torch.as_tensor(self.rew[idx], dtype=torch.float32, device=DEVICE)
        nobs = torch.as_tensor(self.nobs[idx], dtype=torch.float32, device=DEVICE)
        done = torch.as_tensor(self.done[idx], dtype=torch.float32, device=DEVICE)

        return obs, act, rew, nobs, done

class QNetwork(nn.Module):
    def __init__(self, in_dim, out_dim, hid=(256, 256)):
        super().__init__()
        # 简单两层 MLP
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid[0]),
            nn.ReLU(),
            nn.Linear(hid[0], hid[1]),
            nn.ReLU(),
            nn.Linear(hid[1], out_dim)
        )

    def forward(self, x):
        return self.net(x)

class DQNAgent:
    def __init__(self, obs_dim, n_actions, lr=2e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay_steps=50_000,
                 target_update_interval=1000, buffer_size=200_000, batch_size=128):

        self.gamma = gamma
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.batch_size = batch_size
        self.target_update_interval = target_update_interval
        self.learn_steps = 0

        self.policy_net = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.target_net = QNetwork(obs_dim, n_actions).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optim = Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size, obs_dim)

    def epsilon(self, step):
        #  eps_start 过渡到 eps_end
        r = min(1.0, step / max(1, self.eps_decay_steps))
        eps = self.eps_start + r * (self.eps_end - self.eps_start)
        return float(eps)

    def act(self, obs, eps):
        # epsilon-greedy
        if random.random() < eps:
            return random.randrange(self.policy_net.net[-1].out_features)

        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            q_values = self.policy_net(obs_t)
            action = q_values.argmax(dim=1).item()
        return int(action)

    def push(self, o, a, r, no, d):
        self.buffer.push(o, a, r, no, d)

    def train_step(self):
        if self.buffer.sz < self.batch_size:
            return 0.0

        obs, act, rew, nobs, done = self.buffer.sample(self.batch_size)
        # 当前 Q
        q_values = self.policy_net(obs)
        q = q_values.gather(1, act.view(-1, 1)).squeeze(1)
        # 目标 Q
        with torch.no_grad():
            next_q_values = self.target_net(nobs)
            next_max_q = next_q_values.max(dim=1).values
            target = rew + (1.0 - done) * self.gamma * next_max_q
        loss = F.smooth_l1_loss(q, target)

        self.optim.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optim.step()

        self.learn_steps += 1
        if self.learn_steps % self.target_update_interval == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())


class MultiAgentIDQN:
    def __init__(self, env, lr=2e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay_steps=50_000,
                 target_update_interval=1000, buffer_size=200_000, batch_size=128):

        self.env = env
        self.a_list = env.possible_agents

        # 每个 agent 各自一个 DQN
        self.obs_dim = {a: env.observation_spaces[a].shape[0] for a in self.a_list}
        self.act_num = {a: env.action_spaces[a].n for a in self.a_list}

        self.ag = {
            a: DQNAgent(
                self.obs_dim[a],
                self.act_num[a],
                lr=lr,
                gamma=gamma,
                eps_start=eps_start,
                eps_end=eps_end,
                eps_decay_steps=eps_decay_steps,
                target_update_interval=target_update_interval,
                buffer_size=buffer_size,
                batch_size=batch_size
            )
            for a in self.a_list
        }

        self.step = 0

    def select_actions(self, obs_dict):
        if not obs_dict:
            return {}

        eps_list = [self.ag[a].epsilon(self.step) for a in obs_dict.keys()]
        eps = float(np.mean(eps_list))

        actions = {}
        for a in obs_dict.keys():
            actions[a] = self.ag[a].act(obs_dict[a], eps)
        return actions

    def store_transitions(self, obs, actions, rewards, next_obs, dones):
        for a in obs.keys():
            self.ag[a].push(
                obs[a],
                actions.get(a, 0),
                rewards.get(a, 0.0),
                next_obs.get(a, obs[a]),
                bool(dones.get(a, False))
            )

    def train_step(self):
        losses = [self.ag[a].train_step() for a in self.a_list]
        return float(np.mean(losses))


def train_multiagent(env_builder, num_episodes=500, max_steps=200,
                     lr=2e-4, gamma=0.99,
                     eps_start=1.0, eps_end=0.05, eps_decay_steps=50_000,
                     target_update_interval=1000, buffer_size=200_000, batch_size=128,
                     seed=42, log_every=10, label="run"):

    set_seed(seed)
    env = env_builder()

    algo = MultiAgentIDQN(
        env,
        lr,
        gamma,
        eps_start,
        eps_end,
        eps_decay_steps,
        target_update_interval,
        buffer_size,
        batch_size
    )

    ep_returns = []
    all_losses = []
    succ_list = []

    for ep in range(1, num_episodes + 1):
        obs, _ = env.reset(seed=seed + ep)
        ep_r = 0.0
        succ = 0

        for t in range(max_steps):
            acts = algo.select_actions(obs)
            if not acts:
                # no more, end
                break
            nobs, rews, terms, truncs, infos = env.step(acts)
            dones = {a: (terms[a] or truncs[a]) for a in env.possible_agents}
            algo.store_transitions(obs, acts, rews, nobs, dones)
            all_losses.append(algo.train_step())

            step_mean_r = float(np.mean(list(rews.values())))
            ep_r += step_mean_r

            obs = nobs
            algo.step += 1

            if all(dones.values()):
                if any(terms.values()):
                    succ = 1
                break

        ep_returns.append(ep_r)
        succ_list.append(succ)

        if ep % log_every == 0:
            avg_r = np.mean(ep_returns[-log_every:])
            avg_succ = np.mean(succ_list[-log_every:])
            print(f"[{label}] ep {ep}/{num_episodes}  avgR={avg_r:.2f}  succ={avg_succ:.2f}")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep_returns)
    ax[0].set_title(f"Reward ({label})")

    if len(succ_list) > 0:
        window = 20
        kernel = np.ones(window) / window
        smooth_succ = np.convolve(succ_list, kernel, mode="same")
        ax[1].plot(smooth_succ)
    ax[1].set_title("Success rate (moving avg)")

    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"training_curves_{label}.png"), dpi=150)
    plt.close(fig)

    return {
        "episode_rewards": ep_returns,
        "losses": all_losses,
        "success_rates": succ_list
    }

In [6]:
set_seed(42)

def build_warehouse_env():
    return WarehouseParallelEnv(grid_size=6, num_agents=3, max_steps=200, seed=42)

warehouse_results = train_multiagent(
    env_builder=build_warehouse_env,
    num_episodes=800,
    max_steps=200,
    lr=3e-4, gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay_steps=80_000,
    target_update_interval=1500, buffer_size=250_000, batch_size=256,
    seed=42, log_every=20, label="warehouse_idqn"
)

import pickle
with open(os.path.join(RESULTS_DIR, "experiment_results_deep_warehouse_idqn.pkl"), "wb") as f:
    pickle.dump(warehouse_results, f)


[warehouse_idqn] ep 20/800  avgR=-310.42  succ=0.00
[warehouse_idqn] ep 40/800  avgR=-329.42  succ=0.00
[warehouse_idqn] ep 60/800  avgR=-322.75  succ=0.00
[warehouse_idqn] ep 80/800  avgR=-331.42  succ=0.00
[warehouse_idqn] ep 100/800  avgR=-293.58  succ=0.00
[warehouse_idqn] ep 120/800  avgR=-321.50  succ=0.00
[warehouse_idqn] ep 140/800  avgR=-316.22  succ=0.05
[warehouse_idqn] ep 160/800  avgR=-301.10  succ=0.10
[warehouse_idqn] ep 180/800  avgR=-279.27  succ=0.05
[warehouse_idqn] ep 200/800  avgR=-305.33  succ=0.00
[warehouse_idqn] ep 220/800  avgR=-314.58  succ=0.00
[warehouse_idqn] ep 240/800  avgR=-287.27  succ=0.10
[warehouse_idqn] ep 260/800  avgR=-281.78  succ=0.05
[warehouse_idqn] ep 280/800  avgR=-265.98  succ=0.05
[warehouse_idqn] ep 300/800  avgR=-296.75  succ=0.00
[warehouse_idqn] ep 320/800  avgR=-306.17  succ=0.00
[warehouse_idqn] ep 340/800  avgR=-310.83  succ=0.00
[warehouse_idqn] ep 360/800  avgR=-431.60  succ=0.05
[warehouse_idqn] ep 380/800  avgR=-395.22  succ=0.

# petting zoo


In [7]:
from pettingzoo.mpe import simple_spread_v3

def build_mpe_env_idqn():
    return simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=80, continuous_actions=False)

mpe_results = train_multiagent(
    env_builder=build_mpe_env_idqn,
    num_episodes=1500,
    max_steps=80,
    lr=3e-4, gamma=0.99,
    eps_start=1.0, eps_end=0.05, eps_decay_steps=120_000,
    target_update_interval=2000, buffer_size=300_000, batch_size=512,
    seed=123, log_every=20, label="mpe_simple_spread"
)

with open(os.path.join(RESULTS_DIR, "experiment_results_deep_mpe.pkl"), "wb") as f:
    pickle.dump(mpe_results, f)


c:\Users\siiih\AppData\Local\Programs\Python\Python312\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
c:\Users\siiih\AppData\Local\Programs\Python\Python312\Lib\site-packages\pettingzoo\utils\conversions.py:144: UserWarning: The `observation_spaces` dictionary is deprecated. Use the `observation_space` function instead.
  warnings.warn(
c:\Users\siiih\AppData\Local\Programs\Python\Python312\Lib\site-packages\pettingzoo\utils\conversions.py:158: UserWarning: The `action_spaces` dictionary is deprecated. Use the `action_space` function instead.
  warnings.warn(


[mpe_simple_spread] ep 20/1500  avgR=-106.05  succ=0.00
[mpe_simple_spread] ep 40/1500  avgR=-108.56  succ=0.00
[mpe_simple_spread] ep 60/1500  avgR=-108.92  succ=0.00
[mpe_simple_spread] ep 80/1500  avgR=-102.01  succ=0.00
[mpe_simple_spread] ep 100/1500  avgR=-104.18  succ=0.00
[mpe_simple_spread] ep 120/1500  avgR=-97.76  succ=0.00
[mpe_simple_spread] ep 140/1500  avgR=-98.14  succ=0.00
[mpe_simple_spread] ep 160/1500  avgR=-96.40  succ=0.00
[mpe_simple_spread] ep 180/1500  avgR=-95.00  succ=0.00
[mpe_simple_spread] ep 200/1500  avgR=-92.63  succ=0.00
[mpe_simple_spread] ep 220/1500  avgR=-92.91  succ=0.00
[mpe_simple_spread] ep 240/1500  avgR=-94.16  succ=0.00
[mpe_simple_spread] ep 260/1500  avgR=-89.82  succ=0.00
[mpe_simple_spread] ep 280/1500  avgR=-91.65  succ=0.00
[mpe_simple_spread] ep 300/1500  avgR=-82.61  succ=0.00
[mpe_simple_spread] ep 320/1500  avgR=-78.50  succ=0.00
[mpe_simple_spread] ep 340/1500  avgR=-80.16  succ=0.00
[mpe_simple_spread] ep 360/1500  avgR=-73.43  s

In [ ]:
# 5) Double DQN 因为上面表现太差了
class DoubleDQNAgent(DQNAgent):
    def train_step(self):
        if self.buffer.sz < self.batch_size:
            return 0.0
        obs, act, rew, nobs, done = self.buffer.sample(self.batch_size)
        q = self.policy_net(obs).gather(1, act.view(-1, 1)).squeeze(1)
        with torch.no_grad():
            # Double DQN: 用policy_net选动作，用target_net评估
            next_actions = self.policy_net(nobs).argmax(dim=1)
            next_q_target = self.target_net(nobs).gather(1, next_actions.view(-1, 1)).squeeze(1)
            target = rew + (1.0 - done) * self.gamma * next_q_target
        loss = F.smooth_l1_loss(q, target)
        self.optim.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optim.step()
        self.learn_steps += 1
        if self.learn_steps % self.target_update_interval == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        return float(loss.item())


In [ ]:
class MultiAgentDoubleDQN(MultiAgentIDQN):
    def __init__(self, env, lr=3e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay_steps=150_000,
                 target_update_interval=2000, buffer_size=350_000, batch_size=512):
        self.env = env
        self.a_list = env.possible_agents
        self.obs_dim = {a: env.observation_spaces[a].shape[0] for a in self.a_list}
        self.act_num = {a: env.action_spaces[a].n for a in self.a_list}
        self.ag = {a: DoubleDQNAgent(self.obs_dim[a], self.act_num[a], lr=lr, gamma=gamma,
                                     eps_start=eps_start, eps_end=eps_end, eps_decay_steps=eps_decay_steps,
                                     target_update_interval=target_update_interval,
                                     buffer_size=buffer_size, batch_size=batch_size) for a in self.a_list}
        self.step = 0

def train_multiagent_double(env_builder, num_episodes=1800, max_steps=80,
                            lr=3e-4, gamma=0.99, seed=222, log_every=20, label="mpe_simple_spread_ddqn"):
    set_seed(seed)
    env = env_builder()
    algo = MultiAgentDoubleDQN(env)
    ep_ret, ls, succ = [], [], []
    for ep in range(1, num_episodes + 1):
        obs, _ = env.reset(seed=seed + ep)
        rsum, s = 0.0, 0
        for t in range(max_steps):
            acts = algo.select_actions(obs)
            if not acts:
                break
            nobs, rews, terms, truncs, infos = env.step(acts)
            dones = {a: terms[a] or truncs[a] for a in env.possible_agents}
            algo.store_transitions(obs, acts, rews, nobs, dones)
            ls.append(algo.train_step())
            rsum += float(np.mean(list(rews.values())))
            obs = nobs
            algo.step += 1
            if all(dones.values()):
                if any(terms.values()):
                    s = 1
                break
        ep_ret.append(rsum)
        succ.append(s)
        if ep % log_every == 0:
            print(f"[{label}] ep {ep}/{num_episodes}  avgR={np.mean(ep_ret[-log_every:]):.2f}  succ={np.mean(succ[-log_every:]):.2f}")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(ep_ret)
    ax[1].plot(np.convolve(succ, np.ones(20)/20, mode='same') if len(succ)>0 else [])
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"training_curves_{label}.png"), dpi=150)
    plt.close(fig)
    return {"episode_rewards": ep_ret, "losses": ls, "success_rates": succ}


In [ ]:
from pettingzoo.mpe import simple_spread_v3

def build_mpe_env_ddqn():
    return simple_spread_v3.parallel_env(N=3, local_ratio=0.5, max_cycles=80, continuous_actions=False)

print(" MPE simple_spread_v3 (Double DQN)...")
mpe_ddqn_results = train_multiagent_double(
    env_builder=build_mpe_env_ddqn,
    num_episodes=1800,
    max_steps=80,
    lr=3e-4,
    gamma=0.99,
    seed=222,
    log_every=20,
    label="mpe_simple_spread_ddqn"
)

import pickle
with open(os.path.join(RESULTS_DIR, "experiment_results_deep_mpe_ddqn.pkl"), "wb") as f:
    pickle.dump(mpe_ddqn_results, f)

print("MPE DDQN completed， results/experiment_results_deep_mpe_ddqn.pkl")


 MPE simple_spread_v3 (Double DQN)...


c:\Users\siiih\AppData\Local\Programs\Python\Python312\Lib\site-packages\pettingzoo\utils\conversions.py:144: UserWarning: The `observation_spaces` dictionary is deprecated. Use the `observation_space` function instead.
  warnings.warn(
c:\Users\siiih\AppData\Local\Programs\Python\Python312\Lib\site-packages\pettingzoo\utils\conversions.py:158: UserWarning: The `action_spaces` dictionary is deprecated. Use the `action_space` function instead.
  warnings.warn(


[mpe_simple_spread_ddqn] ep 20/1800  avgR=-101.87  succ=0.00
[mpe_simple_spread_ddqn] ep 40/1800  avgR=-112.07  succ=0.00
[mpe_simple_spread_ddqn] ep 60/1800  avgR=-104.28  succ=0.00
[mpe_simple_spread_ddqn] ep 80/1800  avgR=-117.58  succ=0.00
[mpe_simple_spread_ddqn] ep 100/1800  avgR=-102.38  succ=0.00
[mpe_simple_spread_ddqn] ep 120/1800  avgR=-107.38  succ=0.00
[mpe_simple_spread_ddqn] ep 140/1800  avgR=-106.40  succ=0.00
[mpe_simple_spread_ddqn] ep 160/1800  avgR=-92.10  succ=0.00
[mpe_simple_spread_ddqn] ep 180/1800  avgR=-90.20  succ=0.00
[mpe_simple_spread_ddqn] ep 200/1800  avgR=-94.41  succ=0.00
[mpe_simple_spread_ddqn] ep 220/1800  avgR=-91.52  succ=0.00
[mpe_simple_spread_ddqn] ep 240/1800  avgR=-95.81  succ=0.00
[mpe_simple_spread_ddqn] ep 260/1800  avgR=-83.99  succ=0.00
[mpe_simple_spread_ddqn] ep 280/1800  avgR=-85.43  succ=0.00
[mpe_simple_spread_ddqn] ep 300/1800  avgR=-81.34  succ=0.00
[mpe_simple_spread_ddqn] ep 320/1800  avgR=-76.15  succ=0.00
[mpe_simple_spread_dd